# 🔄 Alap ügynök munkafolyamatok GitHub modellekkel (Python)

## 📋 Munkafolyamat-összehangolási oktatóanyag

Ez a jegyzetfüzet bemutatja a Microsoft Agent Framework hatékony **Workflow Builder** képességeit. Tanulja meg, hogyan hozhat létre kifinomult, több lépésből álló ügynök munkafolyamatokat, amelyek képesek kezelni összetett üzleti folyamatokat, és zökkenőmentesen koordinálni több MI műveletet.

## 🎯 Tanulási célok

### 🏗️ **Munkafolyamat architektúra**
- **Workflow Builder**: Tervezze és hangolja össze az összetett, több lépéses folyamatokat
- **Eseményvezérelt végrehajtás**: Kezelje a munkafolyamat eseményeit és állapotváltozásait
- **Vizuális munkafolyamat-tervezés**: Hozzon létre és ábrázoljon munkafolyamat-struktúrákat
- **GitHub modellek integrációja**: Használja ki az MI modelleket a munkafolyamatok kontextusában

### 🔄 **Folyamat összehangolás**
- **Szekvenciális műveletek**: Több ügynöki feladat logikus sorrendbe fűzése
- **Feltételes logika**: Döntési pontok és elágazó munkafolyamatok megvalósítása
- **Hibakezelés**: Robusztus hibajavítás és munkafolyamat-tartósság
- **Állapotkezelés**: Kövesse nyomon és kezelje a munkafolyamat végrehajtási állapotát

### 📊 **Vállalati munkafolyamat minták**
- **Üzleti folyamat automatizálás**: Összetett szervezeti munkafolyamatok automatizálása
- **Több ügynök koordinációja**: Több specializált ügynök összehangolása
- **Skálázható végrehajtás**: Munkafolyamatok tervezése vállalati méretű műveletekhez
- **Monitorozás és megfigyelhetőség**: A munkafolyamat teljesítményének és eredményeinek nyomon követése

## ⚙️ Előfeltételek és beállítás

### 📦 **Szükséges függőségek**

Telepítse az ügynök keretrendszert munkafolyamat képességekkel:

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub modellek konfigurálása**

**Környezeti beállítás (.env fájl):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 **Vállalati felhasználási esetek**

**Üzleti folyamat példák:**
- **Ügyfélbevezetés**: Többlépéses ellenőrzési és beállítási munkafolyamatok
- **Tartalmi folyamat**: Automatizált tartalom létrehozás, felülvizsgálat és publikálás
- **Adatfeldolgozás**: ETL munkafolyamatok MI-alapú átalakítással
- **Minőségbiztosítás**: Automatizált tesztelési és érvényesítési folyamatok

**Munkafolyamat előnyei:**
- 🎯 **Megbízhatóság**: Determinisztikus végrehajtás hibajavítással
- 📈 **Skálázhatóság**: Nagy volumenű folyamat automatizálás kezelése
- 🔍 **Megfigyelhetőség**: Teljes audit naplók és monitorozás
- 🔧 **Karbantarthatóság**: Vizuális tervezés és moduláris komponensek

## 🎨 Munkafolyamat tervezési minták

### Alap munkafolyamat struktúra
```mermaid
graph TD
    A[Indítás] --> B[Agent Feladat 1]
    B --> C{Döntési Pont}
    C -->|Siker| D[Agent Feladat 2]
    C -->|Hiba| E[Hibakezelő]
    D --> F[Vége]
    E --> F
```

**Fő összetevők:**
- **WorkflowBuilder**: Fő összehangoló motor
- **WorkflowEvent**: Eseménykezelés és kommunikáció
- **WorkflowViz**: Vizuális munkafolyamat megjelenítés és hibakeresés

Építsük meg az első intelligens munkafolyamatot! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = await provider.create_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = await provider.create_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
